In [ ]:
"""
Project: ABMS-WP - Agent-Based Modeling for Water Demand Forecasting
Description: Analyzes profile transitions in Scenario CVIII. 
             Generates two distinct reports: 
             1. Sorted by overall volatility (total changes).
             2. Sorted by wasteful behavior shift (risk areas).
"""

import pandas as pd
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================
BASE_PATH = r'C:\Projetos\ABM-WP'
INPUT_FILE = os.path.join(BASE_PATH, 'resultados', 'comparativo_perfil_cenarios_qgis.csv')

# Defining two output files
OUTPUT_TOTAL = os.path.join(BASE_PATH, 'resultados', 'sector_shifts_total_CVIII.csv')
OUTPUT_WASTEFUL = os.path.join(BASE_PATH, 'resultados', 'sector_shifts_wasteful_CVIII.csv')

def analyze_shifts():
    print(f"--- Loading: {INPUT_FILE} ---")
    
    if not os.path.exists(INPUT_FILE):
        print(f"Error: File {INPUT_FILE} not found. Check the path.")
        return

    # Loading the dataset (GAMA Output)
    df = pd.read_csv(INPUT_FILE, sep=',', low_memory=False)
    
    # Cleaning headers: removing single quotes and spaces
    df.columns = [col.replace("'", "").strip() for col in df.columns]

    # ==========================================
    # 2. Filtering and Logic Definition
    # ==========================================
    # Filtering for target scenario: CVIII
    scenario_df = df[df['NM_CENARIO'] == 'CVIII'].copy()
    
    if scenario_df.empty:
        print("Warning: Scenario CVIII not found in the dataset.")
        return

    # Identify any profile change (Volatility)
    scenario_df['Has_Changed'] = scenario_df['TP_COMPORTAMENTO'] != scenario_df['TP_NOVO_COMPORTAMENTO']
    
    # Identify changes TO 'PERDULARIO' (Wasteful Shift)
    scenario_df['To_Wasteful'] = (scenario_df['Has_Changed']) & (scenario_df['TP_NOVO_COMPORTAMENTO'] == 'PERDULARIO')

    # ==========================================
    # 3. Aggregation by Sector (CD_SETOR)
    # ==========================================
    summary = scenario_df.groupby('CD_SETOR').agg(
        Total_Connections=('SK_MATRICULA', 'count'),
        Total_Changes=('Has_Changed', 'sum'),
        Changes_To_Wasteful=('To_Wasteful', 'sum')
    ).reset_index()

    # Calculate rates for academic context
    summary['Change_Rate_%'] = (summary['Total_Changes'] / summary['Total_Connections'] * 100).round(2)
    summary['Wasteful_Shift_Rate_%'] = (summary['Changes_To_Wasteful'] / summary['Total_Connections'] * 100).round(2)

    # ==========================================
    # 4. Sorting and Exporting - REPORT 1 (Total Volatility)
    # ==========================================
    report_total = summary.sort_values(by=['Total_Changes', 'Change_Rate_%'], ascending=False)
    report_total.to_csv(OUTPUT_TOTAL, index=False)
    
    print("\n" + "="*60)
    print("REPORT 1: SECTORS WITH HIGHEST OVERALL VOLATILITY")
    print("="*60)
    print(report_total.head(10).to_string(index=False))

    # ==========================================
    # 5. Sorting and Exporting - REPORT 2 (Wasteful Shift)
    # ==========================================
    report_wasteful = summary.sort_values(by=['Changes_To_Wasteful', 'Wasteful_Shift_Rate_%'], ascending=False)
    report_wasteful.to_csv(OUTPUT_WASTEFUL, index=False)
    
    print("\n" + "="*60)
    print("REPORT 2: SECTORS WITH HIGHEST SHIFT TO WASTEFUL PROFILE")
    print("="*60)
    print(report_wasteful.head(10).to_string(index=False))
    
    print("\n" + "-"*60)
    print(f"Files saved in: {os.path.join(BASE_PATH, 'resultados')}")

if __name__ == "__main__":
    analyze_shifts()

--- Loading: C:\Projetos\ABM-WP\resultados\comparativo_perfil_cenarios_qgis.csv ---
Calculating shifts per census tract...

TOP 10 SECTORS WITH MOST PROFILE SHIFTS (SCENARIO CVIII)
CD_SETOR  Total_Connections  Total_Changes  Changes_To_Wasteful  Change_Rate_%  Wasteful_Shift_Rate_%
    S029                554             77                   34          13.90                   6.14
    S048                453             73                   37          16.11                   8.17
    S027                655             65                   35           9.92                   5.34
    S023                512             54                   20          10.55                   3.91
    S080                329             50                   28          15.20                   8.51
    S078                370             48                   22          12.97                   5.95
    S099                322             48                   22          14.91                   6.83
   